In [1]:
## pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu11
##pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121



In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch

In [3]:
# Vérifier si un GPU est disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation de : {device}")


Utilisation de : cuda


In [4]:
data = pd.read_csv("data/dataset_annotated_balanced.csv")

In [5]:
# Créer un vectorizer TF-IDF
vectorizer = TfidfVectorizer()

In [6]:
# Transformer les blagues en vecteurs TF-IDF
X_tfidf = vectorizer.fit_transform(data["Joke_Normalized"])

In [7]:
X = X_tfidf  
y = data["Offensive_Label"]  

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
#SVM
svm_model = SVC(kernel="linear", C=1.0)
svm_model.fit(X_train, y_train)

SVC(kernel='linear')

In [10]:
# Prédictions sur le jeu de test
y_pred = svm_model.predict(X_test)

# Calculer l'accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy du SVM : {accuracy:.4f}")

# Afficher un rapport de classification détaillé
print("\nRapport de classification :\n", classification_report(y_test, y_pred))

# Afficher la matrice de confusion
print("\nMatrice de confusion :\n", confusion_matrix(y_test, y_pred))

Accuracy du SVM : 0.9521

Rapport de classification :
               precision    recall  f1-score   support

           0       0.97      0.93      0.95      1461
           1       0.93      0.98      0.95      1461

    accuracy                           0.95      2922
   macro avg       0.95      0.95      0.95      2922
weighted avg       0.95      0.95      0.95      2922


Matrice de confusion :
 [[1356  105]
 [  35 1426]]


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

In [12]:
# -------------------------------
# Partie RNN et LSTM avec TF-IDF
# -------------------------------
    
class TextDataset(Dataset):
    def __init__(self, X, y):
        # Conversion de la matrice sparse en tableau dense
        self.X = torch.tensor(X.toarray(), dtype=torch.float32)
        # Conversion de la Series en tableau numpy avant de créer le tenseur
        self.y = torch.tensor(y.values, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [13]:
# Création des DataLoaders pour RNN/LSTM
train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


In [14]:
# Définition des modèles RNN et LSTM
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # x: (batch_size, input_size) -> on ajoute une dimension séquentielle
        x = x.unsqueeze(1)  # devient (batch_size, 1, input_size)
        out, hidden = self.rnn(x)
        # hidden: (1, batch_size, hidden_size)
        return self.fc(hidden.squeeze(0))

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # (batch_size, 1, input_size)
        out, (hidden, cell) = self.lstm(x)
        return self.fc(hidden.squeeze(0))

In [15]:

# Initialisation des modèles
input_size = X_train.shape[1]
hidden_size = 128
output_size = len(set(y))

rnn_model = RNNModel(input_size, hidden_size, output_size).to(device)
lstm_model = LSTMModel(input_size, hidden_size, output_size).to(device)

In [16]:
# -------------------------------
# Partie BERT
# -------------------------------

print("\nPréparation du modèle BERT")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Pour BERT, on sépare le texte brut et les labels en ensembles d'entraînement et de test
texts = data["Joke_Normalized"].tolist()
labels = data["Offensive_Label"].tolist()

X_train_texts, X_test_texts, y_train_texts, y_test_texts = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        # Retourne input_ids, attention_mask et le label
        return (
            encoding['input_ids'].squeeze(0),
            encoding['attention_mask'].squeeze(0),
            self.labels[idx]
        )

# Création des DataLoaders pour BERT
train_dataset_bert = BertDataset(X_train_texts, y_train_texts, tokenizer)
test_dataset_bert = BertDataset(X_test_texts, y_test_texts, tokenizer)
train_loader_bert = DataLoader(train_dataset_bert, batch_size=16, shuffle=True)
test_loader_bert = DataLoader(test_dataset_bert, batch_size=16)


Préparation du modèle BERT


In [17]:
# Définition du modèle BERT
class BertClassifier(nn.Module):
    def __init__(self, output_size):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.fc = nn.Linear(768, output_size)
    
    def forward(self, input_ids, attention_mask):
        # return_dict=False permet de récupérer un tuple : (last_hidden_state, pooled_output)
        _, pooled_output = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        return self.fc(pooled_output)

bert_classifier = BertClassifier(output_size).to(device)
optimizer_bert = optim.Adam(bert_classifier.parameters(), lr=2e-5)

In [18]:
# -------------------------------
# Fonction d'entraînement
# -------------------------------

# Définir la fonction de perte
criterion = nn.CrossEntropyLoss()

def train_model(model, train_loader, optimizer, epochs=5, bert_mode=False):
    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            optimizer.zero_grad()
            if bert_mode:
                input_ids, attention_mask, labels = batch
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = labels.to(device)
                outputs = model(input_ids, attention_mask)
            else:
                X_batch, y_batch = batch
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                outputs = model(X_batch)
                labels = y_batch
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


In [19]:
# Définition du modèle BERT
class BertClassifier(nn.Module):
    def __init__(self):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.fc = nn.Linear(768, output_size)
    
    def forward(self, input_ids, attention_mask):
        _, pooled_output = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        return self.fc(pooled_output)

bert_classifier = BertClassifier().to(device)
optimizer_bert = optim.Adam(bert_classifier.parameters(), lr=2e-5)
# train_model(bert_classifier, train_loader_bert, optimizer_bert, epochs=3, bert_mode=True)


In [ ]:
# -------------------------------
# Entraînement des modèles
# -------------------------------

print("\nEntraînement du modèle BERT")
#train_model(bert_classifier, train_loader_bert, optimizer_bert, epochs=3, bert_mode=True)
train_model(bert_classifier, train_loader_bert, optimizer_bert, epochs=5, bert_mode=True) #stabliser à 4 epoche


print("\nEntraînement du modèle RNN")
optimizer_rnn = optim.Adam(rnn_model.parameters(), lr=0.001)
train_model(rnn_model, train_loader, optimizer_rnn, epochs=5) #stabliser à 4 epoche

print("\nEntraînement du modèle LSTM")
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)
train_model(lstm_model, train_loader, optimizer_lstm, epochs=5) #stabliser à 3 epoche

print("\nVérification du GPU :", device)


Entraînement du modèle BERT
Epoch 1, Loss: 0.0037
Epoch 2, Loss: 0.1320
Epoch 3, Loss: 0.1393
Epoch 4, Loss: 0.0002
Epoch 5, Loss: 0.0011

Entraînement du modèle RNN
Epoch 1, Loss: 0.0450
Epoch 2, Loss: 0.0003
Epoch 3, Loss: 0.0109
Epoch 4, Loss: 0.0001
Epoch 5, Loss: 0.0004

Entraînement du modèle LSTM
Epoch 1, Loss: 0.0005
Epoch 2, Loss: 0.0006
Epoch 3, Loss: 0.0001
Epoch 4, Loss: 0.0012
Epoch 5, Loss: 0.0011

Vérification du GPU : cuda


In [24]:

# -------------------------------
# Évaluation des modèles RNN, LSTM et BERT
# -------------------------------

def evaluate_model(model, test_loader, model_name):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(y_batch.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
    print(f"\nÉvaluation du modèle {model_name}")
    print(classification_report(y_true, y_pred))

    # Afficher la matrice de confusion
    print("\nMatrice de confusion :\n")
    print(confusion_matrix(y_true, y_pred))

# Évaluation des modèles RNN et LSTM
evaluate_model(rnn_model, test_loader, "RNN")
evaluate_model(lstm_model, test_loader, "LSTM")

def evaluate_bert(model, test_loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in test_loader:
            if len(batch) == 3:
                input_ids, attention_mask, labels = batch
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = labels.to(device)
                outputs = model(input_ids, attention_mask)
                _, predicted = torch.max(outputs, 1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(predicted.cpu().numpy())
            else:
                print("Mauvais format de batch dans test_loader")
    print("\nÉvaluation du modèle BERT")
    print(classification_report(y_true, y_pred))
        # Afficher la matrice de confusion
    print("\nMatrice de confusion :\n")
    print(confusion_matrix(y_true, y_pred))

# Évaluation du modèle BERT
evaluate_bert(bert_classifier, test_loader_bert)




Évaluation du modèle RNN
              precision    recall  f1-score   support

           0       0.89      0.84      0.86      1461
           1       0.85      0.90      0.87      1461

    accuracy                           0.87      2922
   macro avg       0.87      0.87      0.87      2922
weighted avg       0.87      0.87      0.87      2922


Matrice de confusion :

[[1227  234]
 [ 152 1309]]

Évaluation du modèle LSTM
              precision    recall  f1-score   support

           0       0.89      0.86      0.87      1461
           1       0.86      0.90      0.88      1461

    accuracy                           0.88      2922
   macro avg       0.88      0.88      0.88      2922
weighted avg       0.88      0.88      0.88      2922


Matrice de confusion :

[[1251  210]
 [ 153 1308]]

Évaluation du modèle BERT
              precision    recall  f1-score   support

           0       0.99      0.93      0.96      1461
           1       0.93      0.99      0.96      1461